<a href="https://colab.research.google.com/github/apk41910/gas_calculator/blob/main/gas_calculator_v4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
R = 0.08206  # L·atm/(mol·K)

from scipy.optimize import brentq

def calc_P(V, n, T):
    P=n*R*T/V
    return P

def calc_V(P, n, T):
    V=n*R*T/P
    return V

def calc_n(P, V, T):
    n=P*V/R/T
    return n

def calc_T(P, V, n):
    T=P*V/n/R
    return T


def to_kelvin(value, unit):
    if unit == "k":
        return value
    elif unit == "c":
        return value + 273.15
    elif unit == "f":
        return (value - 32) * 5/9 + 273.15
    elif unit == "r":
        return value * 5/9


def to_base(var):
    number, unit = values[var]

    if var == "P":
        return number * to_atm[unit]
    elif var == "V":
        return number * to_L[unit]
    elif var == "n":
        return number * to_mol[unit]
    elif var == "T":
        return to_kelvin(number, unit)



to_atm = {
    "atm":  1,
    "pa":   1/101325,
    "kpa":  1/101.325,
    "mpa":  1/0.101325,
    "bar":  1/1.01325,
    "mmhg": 1/760,
    "torr": 1/760,
    "psi":  1/14.696,
  }

# 부피 → L
to_L = {
    "l":   1,
    "ml":  0.001,
    "cm3": 0.001,
    "m3":  1000,
    "ft3": 28.317,
    "gal": 3.7854,
}

# 몰수 → mol
to_mol = {
    "mol":  1,
    "mmol": 0.001,
    "kmol": 1000,
}


to_T = {
    "k": 1,
    "c": 1,
    "f": 1,
    "r": 1
}

# Tc, Pc: 임계온도(K), 임계압력(atm)
# omega: 이심인자(acentric factor)
# M: 분자량(g/mol)

substances = {
    "co2":  {"name": "CO2",  "M": 44.01, "Tc": 304.2, "Pc": 72.9,  "omega": 0.225},
    "n2":   {"name": "N2",   "M": 28.01, "Tc": 126.2, "Pc": 33.5,  "omega": 0.040},
    "o2":   {"name": "O2",   "M": 32.00, "Tc": 154.6, "Pc": 49.8,  "omega": 0.022},
    "ch4":  {"name": "CH4",  "M": 16.04, "Tc": 190.6, "Pc": 45.4,  "omega": 0.011},
    "h2o":  {"name": "H2O",  "M": 18.02, "Tc": 647.1, "Pc": 217.8, "omega": 0.345},
    "nh3":  {"name": "NH3",  "M": 17.03, "Tc": 405.5, "Pc": 111.3, "omega": 0.250},
    "h2":   {"name": "H2",   "M": 2.016, "Tc": 33.2,  "Pc": 12.8,  "omega": -0.216},
    "c2h6": {"name": "C2H6", "M": 30.07, "Tc": 305.3, "Pc": 48.1,  "omega": 0.099},
}

In [ ]:
class EOS:
    name = "EOS"

    def __init__(self, sub):
        self.sub = sub

    def ab(self):
        raise NotImplementedError

    def alpha(self, T):
        return 1.0

    def pressure(self, Vm, T):
        raise NotImplementedError

    def solve_Vm(self, P_target, T):
        return brentq(lambda Vm: self.pressure(Vm, T) - P_target, 0.05, 1000.0)

    def solve_T(self, P_target, Vm):
        return brentq(lambda T: self.pressure(Vm, T) - P_target, 1, 5000)


class Ideal(EOS):
    name = "Ideal"

    def pressure(self, Vm, T):
        return R*T/Vm

class VdW(EOS):
    name = "vdW"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 27*R**2*Tc**2/(64*Pc)
        b = R*Tc/(8*Pc)
        return a, b

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a/Vm**2
class RK(EOS):
    name = "RK"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        c = 2 ** (1/3)
        a = 1 / (9 * (c - 1)) * R**2 * Tc**2.5 / Pc
        b = (c - 1) / 3 * R * Tc / Pc        # ← 추가
        return a, b                           # ← 추가

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a/(T**0.5 * Vm * (Vm + b))



class SRK(EOS):
    name = "SRK"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 0.42748*R**2*Tc**2/Pc
        b = 0.08664*R*Tc/Pc
        return a, b

    def alpha(self, T):
        Tc = self.sub["Tc"]
        omega = self.sub["omega"]
        m = 0.480 + 1.574*omega - 0.176*omega**2
        return (1 + m*(1 - (T/Tc)**0.5))**2

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a*self.alpha(T)/(Vm*(Vm + b))

class PR(EOS):
    name = "PR"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 0.45724*R**2*Tc**2/Pc
        b = 0.07780*R*Tc/Pc
        return a, b

    def alpha(self, T):
        Tc = self.sub["Tc"]
        omega = self.sub["omega"]
        kappa = 0.37464 + 1.54226*omega - 0.26992*omega**2
        return (1 + kappa*(1 - (T/Tc)**0.5))**2

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a*self.alpha(T)/(Vm*(Vm + b) + b*(Vm - b))




Ideal       49.2360 atm
vdW         39.4211 atm
RK          38.4481 atm
SRK         38.3861 atm
PR          37.7033 atm


In [ ]:
while True:
    name = input("물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): ").lower()

    if name == "q":
        break

    if name not in substances:
        print("등록되지 않은 물질입니다.")
        continue

    sub = substances[name]
    print(f"{sub['name']} 선택됨")

    print("사용 가능한 단위")
    print("  압력: atm, kPa, bar, mmHg, psi ...")
    print("  부피: L, mL, m3 ...")
    print("  몰수: mol, mmol, kmol")
    print("  온도: K, C, F")
    eos_list = [Ideal(sub), VdW(sub), RK(sub), SRK(sub), PR(sub)]

    while True:                                # 안쪽: 계산 반복
        text = input("값 3개 입력 (b: 뒤로가기) > ")

        if text == "b":
            break                              # 안쪽만 종료 → 물질 선택으로

        values = {}
        has_error = False

        for item in text.split(","):
            try:
                number, unit = item.split()
                number = float(number)
                unit = unit.lower()

                if unit in to_atm:
                    values["P"] = (number, unit)
                elif unit in to_L:
                    values["V"] = (number, unit)
                elif unit in to_mol:
                    values["n"] = (number, unit)
                elif unit in to_T:
                    values["T"] = (number, unit)
                else:
                    print("모르는 단위입니다:", unit)
            except ValueError:
                print("입력 형식을 확인해주세요. 예: 200 kPa, 2 mol, 300 K")
                has_error = True
                break

        if has_error:
            continue

        missing = [v for v in ["P", "V", "n", "T"] if v not in values]

        if len(missing) != 1:
            print("값 3개를 입력해주세요.")
            continue

        target = missing[0]
        print("{}를 구하겠습니다".format(target))

        if target == "P":
            V, n, T = to_base("V"), to_base("n"), to_base("T")
            Vm = V / n
            for eos in eos_list:
                print(f"{eos.name:<8} P = {eos.pressure(Vm, T):>10.4f} atm")

        elif target == "V":
            P, n, T = to_base("P"), to_base("n"), to_base("T")
            for eos in eos_list:
                print(f"{eos.name:<8} V = {eos.solve_Vm(P, T)*n:>10.4f} L")

        elif target == "n":
            P, V, T = to_base("P"), to_base("V"), to_base("T")
            for eos in eos_list:
                print(f"{eos.name:<8} n = {V/eos.solve_Vm(P, T):>10.4f} mol")

        elif target == "T":
            P, V, n = to_base("P"), to_base("V"), to_base("n")
            Vm = V / n
            for eos in eos_list:
                print(f"{eos.name:<8} T = {eos.solve_T(P, Vm):>10.4f} mol")

물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): co2
CO2 선택됨
사용 가능한 단위
  압력: atm, kPa, bar, mmHg, psi ...
  부피: L, mL, m3 ...
  몰수: mol, mmol, kmol
  온도: K, C, F
값 3개 입력 (b: 뒤로가기) > 37.7 atm, 0.5 L, 1 mol
T를 구하겠습니다
Ideal    T =   229.7100 mol
vdW      T =   290.4110 mol
RK       T =   296.2189 mol
SRK      T =   296.7722 mol
PR       T =   299.9840 mol
값 3개 입력 (b: 뒤로가기) > b
물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): q
